# 04e. Veto de mãe

Censo sem CPF no [`04_atribuir.ipynb`](04_atribuir.ipynb) cujo **melhor par**
cai no veto de mãe: as duas mães preenchidas, nem iguais nem prefixo
(`n ≥ 2` no começo), e não é o caso de nome completo e data iguais.

O [`04c_decompor_sem_cpf.ipynb`](04c_decompor_sem_cpf.ipynb) mistura isso com
primeiro nome ausente ou com mais de 1 de Levenshtein. Aqui os dois ficam
separados. A escada já aceita Jaro-Winkler da mãe ≥ 0,75, e de 0,95 a 0,99
não aplica esses vetos. Este notebook continua mostrando o corte estrito.


In [ ]:
import sys
from pathlib import Path

from IPython.display import display

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    SPLINK_ATRIBUICAO,
    SPLINK_INPUT_VIEW,
    SPLINK_PREDICTIONS,
    TABELA_CENSO_LIMPA_APLICACAO,
    TABELA_CPF_LIMPA_APLICACAO,
    drop_splink_temp_tables,
    get_connection,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

print_paths()
require_input(SPLINK_PREDICTIONS, label='SPLINK_PREDICTIONS (rode o 02b antes)')
require_input(SPLINK_ATRIBUICAO, label='SPLINK_ATRIBUICAO (rode o 04_atribuir antes)')
con = get_connection()
drop_splink_temp_tables(con)
require_tables(
    con,
    [TABELA_CENSO_LIMPA_APLICACAO, TABELA_CPF_LIMPA_APLICACAO],
    notebook_origem='00b',
)
materialize_splink_input(
    con,
    censo_table=TABELA_CENSO_LIMPA_APLICACAO,
    cpf_table=TABELA_CPF_LIMPA_APLICACAO,
)


In [ ]:
con.execute(f'''
CREATE OR REPLACE VIEW splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_censo,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_cpf,
    match_probability
FROM read_parquet('{SPLINK_PREDICTIONS}')
''')

con.execute(f'''
CREATE OR REPLACE VIEW atribuicao_2 AS
SELECT unique_id_censo
FROM read_parquet('{SPLINK_ATRIBUICAO}')
''')

print(
    'Pares no parquet:',
    f"{con.execute('SELECT COUNT(*) FROM splink_predictions').fetchone()[0]:,}",
)
print(
    'Atribuições:',
    f"{con.execute('SELECT COUNT(*) FROM atribuicao_2').fetchone()[0]:,}",
)


Melhor par de quem não saiu no 04. `veto_mae` exige primeiro
nome preenchido e com no máximo 1 de Levenshtein: se o primeiro nome já
reprova, o par vai para `so_primeiro` e não entra na leitura da mãe.


In [ ]:
tok_c = "string_split(ca.nome_mae_phon, ' ')"
tok_p = "string_split(pb.nome_mae_phon, ' ')"
n_mae = f"least(len({tok_c}), len({tok_p}))"
prefixo_mae = (
    f"{n_mae} >= 2 AND "
    f"list_slice({tok_c}, 1, {n_mae}) = list_slice({tok_p}, 1, {n_mae})"
)
# Nome completo e data iguais passam mesmo com mãe diferente.
excecao = (
    "ca.nome_completo_phon IS NOT NULL AND pb.nome_completo_phon IS NOT NULL "
    "AND ca.nome_completo_phon = pb.nome_completo_phon "
    "AND ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL "
    "AND ca.data_nascimento = pb.data_nascimento"
)

con.execute(f'''
CREATE OR REPLACE TABLE censo_sem_cpf AS
SELECT c.unique_id
FROM {SPLINK_INPUT_VIEW} c
WHERE c.origem = 'censo'
  AND NOT EXISTS (
      SELECT 1 FROM atribuicao_2 a WHERE a.unique_id_censo = c.unique_id
  )
''')

con.execute('''
CREATE OR REPLACE TABLE melhor_par AS
SELECT unique_id_censo, unique_id_cpf, match_probability
FROM splink_predictions
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY unique_id_censo
    ORDER BY match_probability DESC, unique_id_cpf
) = 1
''')

con.execute(f'''
CREATE OR REPLACE TABLE melhor_sem AS
SELECT
    s.unique_id AS unique_id_censo,
    p.unique_id_cpf,
    p.match_probability,
    CASE
        WHEN p.unique_id_cpf IS NULL THEN 'sem_par'
        WHEN ca.primeiro_nome_phon IS NULL
             OR pb.primeiro_nome_phon IS NULL
             OR levenshtein(ca.primeiro_nome_phon, pb.primeiro_nome_phon) > 1
            THEN 'so_primeiro'
        WHEN ca.nome_mae_phon IS NOT NULL
             AND pb.nome_mae_phon IS NOT NULL
             AND ca.nome_mae_phon <> pb.nome_mae_phon
             AND NOT ({prefixo_mae})
             AND NOT ({excecao})
            THEN 'veto_mae'
        ELSE 'passaria'
    END AS porta
FROM censo_sem_cpf s
LEFT JOIN melhor_par p ON p.unique_id_censo = s.unique_id
LEFT JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = s.unique_id
LEFT JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
''')

display(con.execute('''
SELECT
    porta,
    COUNT(*) AS n,
    SUM(CASE WHEN match_probability >= 0.75 THEN 1 ELSE 0 END) AS n_p075,
    ROUND(AVG(match_probability), 3) AS avg_p
FROM melhor_sem
GROUP BY 1
ORDER BY n DESC
''').df())


No veto de mãe, como os dois nomes da mãe se afastam, ao lado
do nome da pessoa e da data. `mesmas_palavras_ordem` e `contem_sem_prefixo`
são o lugar em que a comparação pode estar rígida demais: as palavras batem,
a regra de prefixo não.


In [ ]:
tok_c = "string_split(ca.nome_mae_phon, ' ')"
tok_p = "string_split(pb.nome_mae_phon, ' ')"
n_c = f"len({tok_c})"
n_p = f"len({tok_p})"
inter = f"len(list_intersect({tok_c}, {tok_p}))"

con.execute(f'''
CREATE OR REPLACE TABLE veto_mae AS
SELECT
    m.unique_id_censo,
    m.unique_id_cpf,
    m.match_probability,
    ca.nome_completo_phon AS nome_censo,
    pb.nome_completo_phon AS nome_cpf,
    ca.nome_mae_phon AS mae_censo,
    pb.nome_mae_phon AS mae_cpf,
    ca.data_nascimento AS dob_censo,
    pb.data_nascimento AS dob_cpf,
    jaro_winkler_similarity(ca.nome_mae_phon, pb.nome_mae_phon) AS jw_mae,
    CASE
        WHEN list_sort({tok_c}) = list_sort({tok_p}) AND {tok_c} <> {tok_p}
            THEN 'mesmas_palavras_ordem'
        WHEN {tok_c} = {tok_p}
            THEN 'mesmas_palavras'
        WHEN {inter} = least({n_c}, {n_p}) AND {n_c} <> {n_p}
            THEN 'contem_sem_prefixo'
        WHEN jaro_winkler_similarity(ca.nome_mae_phon, pb.nome_mae_phon) >= 0.92
            THEN 'parecidas'
        WHEN {inter} >= 1
            THEN 'alguma_palavra'
        ELSE 'sem_palavra'
    END AS mae_dif,
    CASE
        WHEN ca.nome_completo_phon = pb.nome_completo_phon THEN 'exato'
        WHEN ca.primeiro_nome_phon = pb.primeiro_nome_phon
             AND ca.ultimo_nome_phon = pb.ultimo_nome_phon THEN 'pontas'
        ELSE 'outro'
    END AS nome_pessoa,
    CASE
        WHEN ca.data_nascimento IS NULL AND pb.data_nascimento IS NULL THEN 'as_duas_nulas'
        WHEN ca.data_nascimento IS NULL OR pb.data_nascimento IS NULL THEN 'uma_nula'
        WHEN ca.data_nascimento = pb.data_nascimento THEN 'exata'
        ELSE 'diferente'
    END AS data_par
FROM melhor_sem m
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = m.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = m.unique_id_cpf
WHERE m.porta = 'veto_mae'
''')

n_excecao = con.execute('''
SELECT COUNT(*) FROM veto_mae
WHERE nome_pessoa = 'exato' AND data_par = 'exata'
''').fetchone()[0]
print('Veto de mãe:', f"{con.execute('SELECT COUNT(*) FROM veto_mae').fetchone()[0]:,}")
print('Nome e data exatos dentro do veto (tem de ser 0):', n_excecao)

display(con.execute('''
SELECT
    mae_dif,
    nome_pessoa,
    data_par,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct,
    ROUND(AVG(match_probability), 3) AS avg_p,
    ROUND(AVG(jw_mae), 3) AS avg_jw_mae
FROM veto_mae
GROUP BY 1, 2, 3
ORDER BY n DESC
''').df())


Amostra. Primeiro as mães com as mesmas palavras ou uma dentro
da outra. Depois as parecidas com nome da pessoa exato. Por último, mãe sem
palavra em comum e nome da pessoa exato: o veto no caso claro.


In [ ]:
cols = '''
    match_probability,
    mae_dif,
    nome_pessoa,
    data_par,
    ROUND(jw_mae, 3) AS jw_mae,
    nome_censo,
    nome_cpf,
    mae_censo,
    mae_cpf,
    dob_censo,
    dob_cpf
'''

display(con.execute(f'''
SELECT {cols}
FROM veto_mae
WHERE mae_dif IN ('mesmas_palavras_ordem', 'mesmas_palavras', 'contem_sem_prefixo')
ORDER BY match_probability DESC
LIMIT 40
''').df())

display(con.execute(f'''
SELECT {cols}
FROM veto_mae
WHERE mae_dif = 'parecidas' AND nome_pessoa = 'exato'
ORDER BY match_probability DESC
LIMIT 40
''').df())

display(con.execute(f'''
SELECT {cols}
FROM veto_mae
WHERE mae_dif = 'sem_palavra' AND nome_pessoa = 'exato'
ORDER BY match_probability DESC
LIMIT 20
''').df())


In [ ]:
con.close()
